In [1]:
import routellm
print(routellm.__file__)

/Users/poorna/Desktop/research_work/.venv/lib/python3.12/site-packages/routellm/__init__.py


In [ ]:
ls /Users/poorna/Desktop/research_work/.venv/lib/python3.12/site-packages/routellm/


In [2]:
from routellm.controller import Controller
import json

/Users/poorna/Desktop/research_work/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [34]:
client = Controller(
    routers=["mf"],
    strong_model="gpt-4o",
    weak_model="gpt-4o-mini",
)

# Loa

In [36]:
import numpy as np
import pandas as pd

# Load only the query column
queries = (
    pd.read_parquet("../datasets/processed/gaia.parquet", columns=["query"])["query"]
    .dropna()
    .astype(str)
    .str.strip()
)


In [37]:
# Quick test: run only first 10 querie
scores = [client.routers["mf"].calculate_strong_win_rate(q) for q in queries]



In [ ]:
from pathlib import Path
import pandas as pd

# Build routing results table
results_df = pd.DataFrame({
    "query": queries,
    "strong_win_rate": scores,
})
results_df["threshold"] = float(threshold)
results_df["route_to_strong"] = results_df["strong_win_rate"] >= float(threshold)

# Save parquet
out_dir = Path("../resultts1/unified_baseline/gaia")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "routellm_scores.parquet"
results_df.to_parquet(out_path, index=False)

print(f"Saved {len(results_df)} rows to: {out_path}")

In [40]:
len(scores)

165

In [41]:
target_pct = 0.25
threshold = float(np.percentile(scores, 100 - (target_pct * 100)))
print(f"Queries scored: {len(scores)}")
print(f"Threshold for {target_pct*100:.0f}% strong model calls: {threshold:.5f}")

Queries scored: 165
Threshold for 25% strong model calls: 0.30765


#### 
Queries scored: 165
Threshold for 25% strong model calls: 0.30762



In [58]:
# SYSTEM_PROMPT = """"You are an expert question-answering system.\n"
#         "\n"
#         "Rules:\n"
#         "- Return ONLY the final answer: a single word, number, value, name, date, or short phrase.\n"
#         "- No explanation, no preamble (e.g. do NOT write 'The answer is ...').\n"
#         "- Numbers: use digits only, with a comma as the thousands separator if needed "
#         "(e.g. 42, 3.14, 1,024).\n"
#         "- Names / words / phrases: return them verbatim (e.g. Paris, New York).\n"
#         "if you dont have access to files return "NO ACCESS"
# """

# results = []
# for i, q in enumerate(queries):
#     response = client.chat.completions.create(
#         model="router-mf-0.30765",
#         messages=[
#             {"role": "system", "content": SYSTEM_PROMPT},  # ← add this line
#             {"role": "user",   "content": q}               # ← same as before
#         ]
#     )
#     results.append({
#         "query"     : q,
#         "model_used": response.model,
#         "answer"    : response.choices[0].message.content,
#         "score"     : scores[i]
#     })
#     print(f"Score: {scores[i]:.3f} → {response.model} → {response.choices[0].message.content}")

Score: 0.131 → gpt-4o-mini-2024-07-18 → egalitarian
Score: 0.114 → gpt-4o-mini-2024-07-18 → 33139, 33140, 33141, 33142, 33143
Score: 0.178 → gpt-4o-mini-2024-07-18 → 1,141
Score: 0.161 → gpt-4o-mini-2024-07-18 → space
Score: 0.346 → gpt-4o-2024-08-06 → 9000
Score: 0.230 → gpt-4o-mini-2024-07-18 → Sorry, I can't view attachments.
Score: 0.340 → gpt-4o-2024-08-06 → 5
Score: 0.275 → gpt-4o-mini-2024-07-18 → 100,000
Score: 0.231 → gpt-4o-mini-2024-07-18 → 11/18/17
Score: 0.225 → gpt-4o-mini-2024-07-18 → 2
Score: 0.167 → gpt-4o-mini-2024-07-18 → 82%
Score: 0.255 → gpt-4o-mini-2024-07-18 → The answer cannot be computed directly as requested. Please provide the PDB file or details on how to access it.
Score: 0.298 → gpt-4o-mini-2024-07-18 → 205-735-4; 259-152-2
Score: 0.274 → gpt-4o-mini-2024-07-18 → Mordecai
Score: 0.218 → gpt-4o-mini-2024-07-18 → "Goodbye, you old friend."
Score: 0.219 → gpt-4o-mini-2024-07-18 → 7
Score: 0.258 → gpt-4o-mini-2024-07-18 → 3.9
Score: 0.330 → gpt-4o-2024-08-06 

In [102]:
import time
import pandas as pd
from pathlib import Path

# Pricing per 1K tokens (as of 2024)
PRICING = {
    "gpt-4o"          : {"input": 0.005,  "output": 0.015},
    "gpt-4o-mini"     : {"input": 0.00015,"output": 0.0006},
}

def get_cost(model, input_tokens, output_tokens):
    key = "gpt-4o-mini" if "mini" in model else "gpt-4o"
    price = PRICING[key]
    return (input_tokens * price["input"] + output_tokens * price["output"]) / 1000

results = []
for i, q in enumerate(queries):
    
    start_time = time.time()                          # ← start timer
    
    response = client.chat.completions.create(
        model="router-mf-0.30765",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": q}
        ]
    )
    
    latency = time.time() - start_time                # ← end timer

    # Token counts from response
    input_tokens  = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens
    total_tokens  = response.usage.total_tokens
    cost          = get_cost(response.model, input_tokens, output_tokens)

    results.append({
        "query"        : q,
        "model_used"   : response.model,
        "predicted"    : response.choices[0].message.content,
        "score"        : scores[i],
        "input_tokens" : input_tokens,
        "output_tokens": output_tokens,
        "total_tokens" : total_tokens,
        "cost_usd"     : round(cost, 6),
        "latency_sec"  : round(latency, 3),
    })
    print(f"Score: {scores[i]:.3f} → {response.model} | "
          f"tokens: {total_tokens} | "
          f"cost: ${cost:.5f} | "
          f"latency: {latency:.2f}s"
          f"predicted: {response.choices[0].message.content}"
          f"score: {scores[i]:.3f}"
          )

Score: 0.131 → gpt-4o-mini-2024-07-18 | tokens: 208 | cost: $0.00003 | latency: 0.77spredicted: egalitarianscore: 0.131
Score: 0.114 → gpt-4o-mini-2024-07-18 | tokens: 250 | cost: $0.00005 | latency: 1.09spredicted: 33154, 33160, 33162, 32118, 92629score: 0.114
Score: 0.178 → gpt-4o-mini-2024-07-18 | tokens: 208 | cost: $0.00003 | latency: 1.32spredicted: 4score: 0.178
Score: 0.161 → gpt-4o-mini-2024-07-18 | tokens: 222 | cost: $0.00003 | latency: 0.72spredicted: gscore: 0.161
Score: 0.346 → gpt-4o-2024-08-06 | tokens: 216 | cost: $0.00110 | latency: 0.92spredicted: 11000score: 0.346
Score: 0.230 → gpt-4o-mini-2024-07-18 | tokens: 181 | cost: $0.00003 | latency: 2.31spredicted: Ocean's 11score: 0.230
Score: 0.340 → gpt-4o-2024-08-06 | tokens: 171 | cost: $0.00086 | latency: 0.86spredicted: 7score: 0.340
Score: 0.275 → gpt-4o-mini-2024-07-18 | tokens: 208 | cost: $0.00003 | latency: 1.09spredicted: 140,000score: 0.275
Score: 0.231 → gpt-4o-mini-2024-07-18 | tokens: 168 | cost: $0.00003 

In [105]:
print(df_gaia.columns.tolist())

['query', 'level', 'ground_truth', 'predicted', 'correct', 'model_used', 'score']


In [109]:
results_df = pd.DataFrame(results)

merged = results_df.merge(
    df[["query", "answer", "level"]],
    on="query", how="left"
).rename(columns={"answer": "ground_truth"})

merged["correct"] = merged["predicted"].str.strip().str.lower() == \
                    merged["ground_truth"].str.strip().str.lower()

merged = merged[[
    "query", "level", "ground_truth", "predicted", "correct",
    "model_used", "score",
    "input_tokens", "output_tokens", "total_tokens",  # ← new
    "cost_usd", "latency_sec"                         # ← new
]]

out_dir = Path("../results1/unified_baseline/gaia")
out_dir.mkdir(parents=True, exist_ok=True)
merged.to_parquet(out_dir / "routellm_final1.parquet", index=False)
merged.to_csv(out_dir / "routellm_final1.csv", index=False)

In [108]:
print(f"Overall Accuracy   : {merged['correct'].mean()*100:.1f}%")
print(f"\nAccuracy by Level  :")
print(merged.groupby("level")["correct"].mean().mul(100).round(1).to_string())

print(f"\n── Token Usage ──────────────────")
print(f"Total input tokens : {merged['input_tokens'].sum():,}")
print(f"Total output tokens: {merged['output_tokens'].sum():,}")
print(f"Total tokens       : {merged['total_tokens'].sum():,}")

print(f"\n── Cost ─────────────────────────")
print(f"Total cost         : ${merged['cost_usd'].sum():.4f}")
print(f"Avg cost/query     : ${merged['cost_usd'].mean():.5f}")
strong_cost = merged[~merged["model_used"].str.contains("mini")]["cost_usd"].sum()
weak_cost   = merged[merged["model_used"].str.contains("mini")]["cost_usd"].sum()
print(f"GPT-4o cost        : ${strong_cost:.4f}")
print(f"GPT-4o-mini cost   : ${weak_cost:.4f}")

print(f"\n── Latency ──────────────────────")
print(f"Avg latency        : {merged['latency_sec'].mean():.2f}s")
print(f"Max latency        : {merged['latency_sec'].max():.2f}s")
print(f"Min latency        : {merged['latency_sec'].min():.2f}s")
print(merged.groupby("model_used")["latency_sec"].mean().round(2).to_string())

Overall Accuracy   : 7.3%

Accuracy by Level  :
level
1     5.7
2    10.5
3     0.0

── Token Usage ──────────────────
Total input tokens : 33,642
Total output tokens: 2,105
Total tokens       : 35,747

── Cost ─────────────────────────
Total cost         : $0.0506
Avg cost/query     : $0.00031
GPT-4o cost        : $0.0457
GPT-4o-mini cost   : $0.0049

── Latency ──────────────────────
Avg latency        : 1.87s
Max latency        : 121.77s
Min latency        : 0.67s
model_used
gpt-4o-2024-08-06         3.85
gpt-4o-mini-2024-07-18    1.20


In [111]:
import json
from pathlib import Path

# ── Compute all metrics ───────────────────────────────
total_queries      = len(merged)
correct_queries    = merged["correct"].sum()
accuracy           = merged["correct"].mean()

total_prompt_tokens     = int(merged["input_tokens"].sum())
total_completion_tokens = int(merged["output_tokens"].sum())
total_tokens            = int(merged["total_tokens"].sum())
avg_tokens_per_query    = round(total_tokens / total_queries, 3)

total_cost_usd     = round(merged["cost_usd"].sum(), 6)
avg_latency_s      = round(merged["latency_sec"].mean(), 3)
p50_latency_s      = round(merged["latency_sec"].quantile(0.50), 3)
p95_latency_s      = round(merged["latency_sec"].quantile(0.95), 3)

strong_cost        = merged[~merged["model_used"].str.contains("mini")]["cost_usd"].sum()
weak_cost          = merged[merged["model_used"].str.contains("mini")]["cost_usd"].sum()

# ── Per level metrics ─────────────────────────────────
gaia_level_metrics = {}
for level in sorted(merged["level"].unique()):
    level_df = merged[merged["level"] == level]
    gaia_level_metrics[str(level)] = {
        "total"   : int(len(level_df)),
        "correct" : int(level_df["correct"].sum()),
        "accuracy": round(level_df["correct"].mean(), 6)
    }

# ── Build JSON ────────────────────────────────────────
metrics = {
    "dataset"                 : "gaia",
    "modality"                : "routellm_mf",
    "strong_model"            : "gpt-4o",
    "weak_model"              : "gpt-4o-mini",
    "router"                  : "mf",
    "threshold"               : 0.30765,
    "system_prompt_version"   : "v1",
    "total_queries"           : total_queries,
    "correct_queries"         : int(correct_queries),
    "accuracy"                : round(accuracy, 6),
    "total_prompt_tokens"     : total_prompt_tokens,
    "total_completion_tokens" : total_completion_tokens,
    "total_tokens"            : total_tokens,
    "avg_tokens_per_query"    : avg_tokens_per_query,
    "total_cost_usd"          : total_cost_usd,
    "strong_model_cost_usd"   : round(strong_cost, 6),
    "weak_model_cost_usd"     : round(weak_cost, 6),
    "avg_latency_s"           : avg_latency_s,
    "p50_latency_s"           : p50_latency_s,
    "p95_latency_s"           : p95_latency_s,
    "error_count"             : int((merged["predicted"] == "NO ACCESS").sum()),
    "error_rate"              : round((merged["predicted"] == "NO ACCESS").mean(), 6),
    "routing_split": {
        "strong_count"  : int((~merged["model_used"].str.contains("mini")).sum()),
        "weak_count"    : int(merged["model_used"].str.contains("mini").sum()),
        "strong_pct"    : round((~merged["model_used"].str.contains("mini")).mean() * 100, 2),
        "weak_pct"      : round(merged["model_used"].str.contains("mini").mean() * 100, 2),
    },
    "metadata": {
        "gaia_level_metrics": gaia_level_metrics
    }
}

# ── Save JSON ─────────────────────────────────────────
out_dir = Path("../results1/unified_baseline/gaia")
out_dir.mkdir(parents=True, exist_ok=True)

json_path = out_dir / "routellm_metrics.json"
with open(json_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved metrics to: {json_path}")
print(json.dumps(metrics, indent=2))

Saved metrics to: ../results1/unified_baseline/gaia/routellm_metrics.json
{
  "dataset": "gaia",
  "modality": "routellm_mf",
  "strong_model": "gpt-4o",
  "weak_model": "gpt-4o-mini",
  "router": "mf",
  "threshold": 0.30765,
  "system_prompt_version": "v1",
  "total_queries": 165,
  "correct_queries": 12,
  "accuracy": 0.072727,
  "total_prompt_tokens": 33642,
  "total_completion_tokens": 2105,
  "total_tokens": 35747,
  "avg_tokens_per_query": 216.648,
  "total_cost_usd": 0.050597,
  "strong_model_cost_usd": 0.04569,
  "weak_model_cost_usd": 0.004907,
  "avg_latency_s": 1.873,
  "p50_latency_s": 0.966,
  "p95_latency_s": 1.675,
  "error_count": 0,
  "error_rate": 0.0,
  "routing_split": {
    "strong_count": 42,
    "weak_count": 123,
    "strong_pct": 25.45,
    "weak_pct": 74.55
  },
  "metadata": {
    "gaia_level_metrics": {
      "1": {
        "total": 53,
        "correct": 3,
        "accuracy": 0.056604
      },
      "2": {
        "total": 86,
        "correct": 9,
      

In [ ]:
import numpy as np

scores_array = np.array(scores)

# Sort to visualize
sorted_scores = np.sort(scores_array)

# Percentile calculation
threshold = np.percentile(sorted_scores, 75)  # 75 = 100 - 25
print(f"Threshold : {threshold:.5f}")         # → 0.30762

# Verify split
above = (scores_array >= threshold).sum()
below = (scores_array <  threshold).sum()
print(f"Above (strong) : {above} ({above/len(scores)*100:.1f}%)")  # → 42 (25.5%)
print(f"Below (weak)   : {below} ({below/len(scores)*100:.1f}%)")  # → 123 (74.5%)

Threshold : 0.30765
Above (strong) : 42 (25.5%)
Below (weak)   : 123 (74.5%)
